# Snowman Image Classifier — Training

This notebook builds an image classifier that can tell the difference between a friendly snowman (Olaf) and a hostile snowman (Marshmallow), both from Disney's *Frozen*. Downloads training images via DuckDuckGo, fine-tunes ResNet-18 using fast.ai, and exports the model for the demo notebook. See the repo README for project background.

## 1. Install and import dependencies

In [33]:
!pip install -q fastai duckduckgo_search

In [34]:
from fastai.vision.all import *
import time

In [40]:
import warnings
warnings.filterwarnings('ignore')

## 2. Download training images

We'll use DuckDuckGo image search to download ~150 images per class. No API key required.

Why so few images? Transfer learning. Rather than training from scratch (which requires thousands of examples), we start with ResNet-18, a model pre-trained on the ILSVRC subset of ImageNet (~1.2 million images). It already understands edges, shapes, and textures. We only need enough new images to teach it the specific distinction we care about.

In [36]:
from duckduckgo_search import DDGS

def search_images_ddg(term, max_images=150):
    """Search DuckDuckGo for images and return a list of URLs."""
    with DDGS() as ddgs:
        results = ddgs.images(term, max_results=max_images)
        return L(r['image'] for r in results)

In [37]:
# Define our two classes and the search terms that will find good training images.
# Adding 'disney frozen' to the Olaf search to avoid generic snowman results.
searches = {
    'olaf': 'olaf disney frozen snowman',
    'hostile': 'frozen marshmallow aggressive snowman'
}

path = Path('snowman')

# Download images for each class into its own subfolder.
# download_images handles failures gracefully: broken URLs are skipped.
# Sleep between searches to avoid DDG rate limiting.
if not path.exists():
    path.mkdir()
    for i, (folder, term) in enumerate(searches.items()):
        if i > 0:
            time.sleep(30) # Increased sleep time to avoid rate limiting
        dest = path/folder
        dest.mkdir(exist_ok=True)
        urls = search_images_ddg(term)
        download_images(dest, urls=urls)
        print(f'Downloaded {len(get_image_files(dest))} images for: {folder}')

In [38]:
# Some downloaded images may be corrupt or in unsupported formats.
# verify_images checks each file and returns a list of bad ones.
fns = get_image_files(path)
failed = verify_images(fns)
failed.map(Path.unlink)
print(f'Removed {len(failed)} corrupt images. {len(get_image_files(path))} remaining.')

Removed 0 corrupt images. 0 remaining.


## 3. Build the DataBlock and inspect the data

The first step in any data problem is to look at the data. We need to understand what we have before we can train on it.

fast.ai's DataBlock API lets us describe how to load and label our data in a flexible, reusable way. We tell it:
- What kind of inputs and outputs we have (images and categories)
- How to split into train/validation sets
- How to get labels (from the parent folder name)
- What transforms to apply (random crops and augmentations to improve generalization)

In [39]:
snowman = DataBlock(
    blocks=(ImageBlock, CategoryBlock),       # inputs are images, outputs are categories
    get_items=get_image_files,                # how to find the files
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # 80/20 train/validation split
    get_y=parent_label,                       # label = parent folder name (olaf / hostile)
    item_tfms=RandomResizedCrop(224, min_scale=0.5),  # randomly crop 50-100% of the image, resize to 224x224
    batch_tfms=aug_transforms()               # random flips, rotations, lighting changes
)

dls = snowman.dataloaders(path)

TypeError: 'NoneType' object is not iterable

In [ ]:
# Sanity check: view a sample batch to confirm labels and image quality.
dls.valid.show_batch(max_n=6, nrows=2)

## 4. Train the model

We use `vision_learner` to create a ResNet-18 model with a new classification head for our two classes.

`fine_tune(4)` runs fast.ai's two-phase training:
1. **Freeze** (1 epoch): pre-trained layers are frozen. Only the new head (the classification layer) is trained. This adapts the output to our specific classes without destroying the pre-trained weights.
2. **Unfreeze** (4 epochs): all layers are unfrozen and trained together, using discriminative learning rates: smaller rates for early layers (which already understand basic features) and larger rates for later layers.

That's 5 total epochs of training.

In [ ]:
# vision_learner creates a CNN using a pre-trained backbone (resnet18).
# error_rate tracks the fraction of validation images classified incorrectly.
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

## 5. Interpret results

The confusion matrix shows where the model makes mistakes. For a binary classifier, we want to see high numbers on the diagonal (correct predictions) and low numbers off-diagonal (errors).

`plot_top_losses` shows the images where the model performed worst, which is useful for spotting mislabeled or ambiguous images in the training set.

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
# Top losses = images with the highest loss, which includes both incorrect
# predictions, and correct predictions where the model had low confidence.
interp.plot_top_losses(5, nrows=1)

## 6. Export the model

`learn.export()` saves the model weights and all preprocessing steps (transforms, label mappings) into a single `export.pkl` file. This is everything needed to run inference, no training code required.

The exported model is used by `snowman_demo.ipynb` for the interactive classifier.

In [ ]:
learn.export()

# Confirm the file was saved
path = Path()
path.ls(file_exts='.pkl')